# 042926 stripe-angle survey across all slices

全180断面について縮小ROIからstripe補正角度を推定し、固定角度とZ方向に平滑化した個別角度のどちらを使うべきか判断します。

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, rotate
import tifffile as tiff

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Project root was not found.')

INPUT_DIR = PROJECT_ROOT / 'outputs' / '042926_MAY08R_FOS_1_retake_c_uint16_scale10000'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'preprocess_angle_series'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / '042926_stripe_angle_estimates.csv'
PLOT_PATH = OUTPUT_DIR / '042926_stripe_angle_estimates.png'
files = sorted(INPUT_DIR.glob('*.tif'))
print('Slices:', len(files))
if len(files) != 180:
    raise RuntimeError('Expected 180 input slices, found {}'.format(len(files)))

In [ ]:
candidate_angles = np.arange(-4.0, 4.0001, 0.025)

def estimate_stripe_angle(image):
    roi = np.asarray(image[1400:3000:4, 400:1500:4], dtype=np.float32)
    highpass = roi - gaussian_filter(roi, sigma=(7.5, 7.5))
    highpass *= np.outer(np.hanning(highpass.shape[0]), np.hanning(highpass.shape[1]))
    scores = []
    for angle in candidate_angles:
        aligned = rotate(
            highpass, angle=angle, reshape=False, order=1,
            mode='constant', cval=0, prefilter=False
        )
        profile = np.median(aligned[30:-30, 30:-30], axis=1)
        scores.append(np.std(profile))
    scores = np.asarray(scores)
    best_index = int(np.argmax(scores))
    confidence = float(scores[best_index] / np.median(scores))
    return float(candidate_angles[best_index]), float(scores[best_index]), confidence

rows = []
started = time.time()
for index, path in enumerate(files, start=1):
    image = tiff.imread(str(path))
    angle, score, confidence = estimate_stripe_angle(image)
    rows.append({
        'slice': index, 'file': path.name, 'angle_deg': angle,
        'alignment_score': score, 'confidence_ratio': confidence,
    })
    if index == 1 or index % 10 == 0 or index == len(files):
        print('{}/{} angle={:.3f} elapsed={:.1f}s'.format(
            index, len(files), angle, time.time() - started
        ))

angles = pd.DataFrame(rows)
angles['recommended_angle_deg'] = angles['angle_deg'].rolling(
    window=11, center=True, min_periods=1
).median()
global_median = float(angles['angle_deg'].median())
angles['global_median_deg'] = global_median
angles.to_csv(str(CSV_PATH), index=False)
print('Global median: {:.3f} degrees'.format(global_median))
print(angles['angle_deg'].describe().to_string())
print('Saved:', CSV_PATH)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), dpi=160, sharex=True)
axes[0].plot(angles['slice'], angles['angle_deg'], '.', ms=3, alpha=0.65, label='per-slice estimate')
axes[0].plot(angles['slice'], angles['recommended_angle_deg'], lw=2, label='recommended: rolling median (11)')
axes[0].axhline(global_median, color='black', ls='--', lw=1, label='global median')
axes[0].set_ylabel('Correction angle (degrees)')
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].plot(angles['slice'], angles['confidence_ratio'], color='tab:green', lw=1)
axes[1].set_xlabel('Slice')
axes[1].set_ylabel('Peak / median score')
axes[1].grid(alpha=0.25)
fig.suptitle('Stripe-angle estimates across 042926 stack')
fig.tight_layout()
fig.savefig(str(PLOT_PATH), bbox_inches='tight')
plt.show()
print('Saved:', PLOT_PATH)